#  Multi-label Classification

**다중 분류 vs 다중 레이블 분류**

| 구분    | 다중 분류 (Multi-class)          | 다중 레이블 분류 (Multi-label)         |
| ----- | ---------------------------- | ------------------------------- |
| 정의    | 하나의 샘플이 여러 클래스 중 **하나**에만 속함 | 하나의 샘플이 **여러 클래스에 동시에** 속할 수 있음 |
| 예시    | 고양이, 개, 새 중 하나               | 영화가 Action + Sci-Fi + Drama     |
| 출력 형태 | 정수 인덱스 (`y=3`)               | 이진 벡터 (`y=[1, 0, 1, 0, 1]`)     |
| 모델 출력 | `argmax` 사용                  | `sigmoid` 후 **각 클래스마다 이진 판단**   |

**다중 레이블 문제의 대표 예시**

* 텍스트 분류 (뉴스 → 여러 주제)
* 영화/음악 장르 분류
* 이미지에서 객체 감지 (여러 객체 포함 가능)
* 질병 진단 (동시 복합 질병)

## MultiLabelBinarizer

In [1]:
import pandas as pd

data = pd.DataFrame({
    'plot': [
        "A man fights crime in a futuristic city.",
        "A love story set in wartime.",
        "Aliens invade Earth and a war begins.",
        "A detective solves a complicated crime case.",
        "A dramatic romance in the midst of a tragedy."
    ],
    'genres': [
        ['Action', 'Sci-Fi'],
        ['Romance', 'Drama'],
        ['Action', 'Sci-Fi', 'War'],
        ['Crime', 'Mystery'],
        ['Drama', 'Romance']
    ]
})
data

,plot,genres
0,A man fights crime in a futuristic city.,"[Action, Sci-Fi]"
1,A love story set in wartime.,"[Romance, Drama]"
2,Aliens invade Earth and a war begins.,"[Action, Sci-Fi, War]"
3,A detective solves a complicated crime case.,"[Crime, Mystery]"
4,A dramatic romance in the midst of a tragedy.,"[Drama, Romance]"


In [2]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   plot    5 non-null      object
 1   genres  5 non-null      object
dtypes: object(2)
memory usage: 208.0+ bytes


In [3]:
# 다중 레이블 전처리
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
y = mlb.fit_transform(data['genres'])
print(y)
print(mlb.classes_)

label_df = pd.DataFrame(y, columns=mlb.classes_, index=data['plot'])
label_df

[[1 0 0 0 0 1 0]
 [0 0 1 0 1 0 0]
 [1 0 0 0 0 1 1]
 [0 1 0 1 0 0 0]
 [0 0 1 0 1 0 0]]
['Action' 'Crime' 'Drama' 'Mystery' 'Romance' 'Sci-Fi' 'War']


,Action,Crime,Drama,Mystery,Romance,Sci-Fi,War
plot,,,,,,,
A man fights crime in a futuristic city.,1,0,0,0,0,1,0
A love story set in wartime.,0,0,1,0,1,0,0
Aliens invade Earth and a war begins.,1,0,0,0,0,1,1
A detective solves a complicated crime case.,0,1,0,1,0,0,0
A dramatic romance in the midst of a tragedy.,0,0,1,0,1,0,0


## 다중레이블 분류 모델

In [4]:
# 입력데이터 전처리
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(data['plot'])

input_df = pd.DataFrame(X.toarray(),
                        columns=vectorizer.get_feature_names_out(),
                        index=data['plot'])
input_df


,aliens,and,begins,case,city,complicated,crime,detective,dramatic,earth,fights,futuristic,in,invade,love,man,midst,of,romance,set,solves,story,the,tragedy,war,wartime
plot,,,,,,,,,,,,,,,,,,,,,,,,,,
A man fights crime in a futuristic city.,0.000000,0.000000,0.000000,0.000000,0.442832,0.000000,0.357274,0.000000,0.000000,0.000000,0.442832,0.442832,0.296570,0.000000,0.000000,0.442832,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
A love story set in wartime.,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.317527,0.000000,0.474125,0.000000,0.000000,0.000000,0.000000,0.474125,0.000000,0.474125,0.000000,0.000000,0.000000,0.474125
Aliens invade Earth and a war begins.,0.408248,0.408248,0.408248,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.408248,0.000000,0.000000,0.000000,0.408248,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.408248,0.000000
A detective solves a complicated crime case.,0.000000,0.000000,0.000000,0.463693,0.000000,0.463693,0.374105,0.463693,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.463693,0.000000,0.000000,0.000000,0.000000,0.000000
A dramatic romance in the midst of a tragedy.,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.393795,0.000000,0.000000,0.000000,0.263729,0.000000,0.000000,0.000000,0.393795,0.393795,0.393795,0.000000,0.000000,0.000000,0.393795,0.393795,0.000000,0.000000


In [ ]:
# 모델
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

# 문제의 정답이 하나의 클래스가 아니라 여러 레이블을 동시에 가지는 구조이므로 
# fit 실행 시 레이블 개수만큼 LogisticRegression 모델이 내부적으로 학습 된다.
clf = OneVsRestClassifier(LogisticRegression())
clf.fit(X,y)

In [ ]:
# 예측
test_plot = ['An allen spaces lands in the middle of a war']

X_test = vectorizer.transform(test_plot)
y_pred = clf.predict(X_test)
y_pred_proba = clf.predict_proba(X_test)
print(y_pred_proba)

# 임계치 조정
y_pred = (y_pred_proba >= 0.4).astype(int)
print(y_pred)
y_pred_label = mlb.inverse_transform(y_pred)
y_pred_label

## RNN기반 다중레이블 분류

In [ ]:
# 데이터준비
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import torch

tokenizer = Tokenizer(oov_token='OOV')
tokenizer.fit_on_texts(data['plot'])
X = tokenizer.texts_to_sequences(data['plot'])
X = pad_sequences(X, maxlen=10)
X = torch.tensor(X, dtype=torch.long)
X

In [ ]:
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(data['genres'])
y = torch.tensor(y, dtype=torch.float)
y

In [ ]:
# 모델 생성
import torch.nn as nn
class MultiLabelNet(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.gru = nn.GRU(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forwaed(self, x):
        x = self.embedding(x)
        _, hidden = self.gru(x)
        output = self.fc(hidden[-1])
        return output

In [ ]:
# 모델 학습
import torch.optim as optim
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 100
hidden_dim = 64
output_dim = len(mlb.classes_)

model = MultiLabelNet(vocab_size, embedding_dim, hidden_dim, output_dim)
criterion = nn.BCEWithLogitsLoss()    # 클래스뱔 sigmoid 사용
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 100
for epoch in range(epochs):
    optimizer.zero_grad()
    output = model(X)
    loss = criterion(output, y)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(f'Epoch ({epoch + 1}/{epoch}): Loss = {loss.item():.4f}')

In [ ]:
# 예측 
test_plot = []